<a href="https://colab.research.google.com/github/psh-computer/Machine-Learning/blob/main/medication_assistant_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 약 복용 정보 안내 프로젝트 — Colab 실행 노트북

텍스트나 약봉지 사진에서 약 정보를 확인하고, 식약처 e약은요/DUR 근거를 검색해 Qwen3-8B가 간결하게 설명하는 교육용 프로토타입입니다.

Google Colab에서 GPU 런타임을 선택한 뒤 위에서부터 실행하세요. Drive에 API 원문과 결과물을 저장합니다. 진단·처방 도구가 아닙니다.

## 0. Colab 환경과 파일 저장 위치

먼저 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요. 이 노트북은 기존 PyTorch/CUDA를 다시 설치하지 않습니다. Google Drive를 연결해 API 원문, QA, 평가 결과, 학습 어댑터를 보존합니다.


In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-kor


### 패키지 설치

설치 셀은 Transformers 5와 Hugging Face Hub 1.x, TRL 1.18, Datasets 4.7을 한 조합으로 맞춥니다. Colab의 Gradio/Diffusers가 요구하는 Hub 1.x, cuDF가 요구하는 PyArrow 24 미만, gcsfs가 사용하는 fsspec 버전도 고려해 고정했습니다. PyTorch와 CUDA 빌드는 건드리지 않습니다.

설치가 끝나면 런타임을 재시작한 다음 노트북을 위에서부터 다시 실행하세요. 이미 여러 번 패키지를 바꿔 설치해 `transformers` 오류가 났다면 이 셀을 실행하기 전에 Colab에서 **런타임 → 세션 다시 시작**을 먼저 하세요.


In [ ]:
%pip uninstall -y -q transformers tokenizers
%pip install -q --no-cache-dir \
    "transformers==5.23.0" \
    "tokenizers>=0.22,<0.24" \
    "huggingface-hub>=1.32,<2" \
    "accelerate>=1.11,<2" \
    "datasets==4.7.0" \
    "pyarrow==23.0.1" \
    "fsspec==2025.12.0" \
    "bitsandbytes==0.50.2" \
    "peft>=0.17,<1" \
    "trl==1.18.0" \
    "sentence-transformers" \
    "pillow<13" \
    "pandas" "requests" "pytesseract" "matplotlib"


In [ ]:
# 패키지 설치 직후에는 런타임을 재시작하고 이 셀부터 다시 실행하세요.
import sys, torch, transformers, huggingface_hub, accelerate, datasets, pyarrow, peft, trl, bitsandbytes
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
print("transformers:", transformers.__version__)
print("huggingface-hub:", huggingface_hub.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__, "pyarrow:", pyarrow.__version__)
print("peft:", peft.__version__, "trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
assert torch.cuda.is_available(), "Colab 런타임 유형을 GPU로 변경하세요."


### Google Drive 연결

파일을 보존할 Drive 폴더를 만들고 작업 경로로 지정합니다. 처음 실행할 때 Drive 접근 권한을 승인하세요.

In [ ]:
from pathlib import Path
import os
PROJECT_DIR = Path("/workspace/medication_assistant")   # 영구 저장되는 볼륨 경로
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
os.environ["HF_HOME"] = "/workspace/hf_cache"           # 모델(약 16GB) 캐시도 볼륨에
for folder in ("data", "outputs", "artifacts"):
    Path(folder).mkdir(exist_ok=True)
print("저장 위치:", Path.cwd())

### 공통 코드

다음 셀 하나에 API 호출, 약 문서 만들기, OCR, DUR 조회, 프롬프트·인용 검증, Qwen3-8B 추론이 들어 있습니다. **베이스라인·SFT·DPO·서비스가 모두 같은 시스템 프롬프트와 같은 함수**를 씁니다.

In [ ]:
"""근거 기반 복약 정보 안내 도우미 (교육용 프로토타입, 진단·처방 도구 아님).
API 키는 환경변수로만 읽는다. 모든 단계(베이스라인/SFT/DPO)가 아래 같은 프롬프트·같은 평가 코드를 쓴다."""
from __future__ import annotations

import html
import json
import os
import random
import re
import time
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import requests

MODEL_ID = "Qwen/Qwen3-8B"

# !! 아래 서비스/오퍼레이션 이름은 공공데이터포털 '참고문서'와 대조하세요 (probe_operation으로 실제 호출도 가능).
EASY_BASE, EASY_OP = "https://apis.data.go.kr/1471000/DrbEasyDrugInfoService", "getDrbEasyDrugList"
APPROVAL_BASE, APPROVAL_OP = "https://apis.data.go.kr/1471000/DrugPrdtPrmsnInfoService08", "getDrugPrdtPrmsnInq08"
DUR_BASE, DUR_OP = "https://apis.data.go.kr/1471000/DURPrdlstInfoService03", "getUsjntTabooInfoList03"
SOURCE_URL = "https://nedrug.mfds.go.kr/pbp/CCBBB01/getItemDetail?itemSeq={item_seq}"   # 브라우저로 열어 확인

# 응답 필드 (!! 실제 응답과 대조 필요 — 틀리면 여기만 고치면 됨)
EASY_SECTIONS = {"효능": "efcyQesitm", "사용법": "useMethodQesitm", "사용 전 주의사항": "atpnWarnQesitm",
                 "사용상 주의사항": "atpnQesitm", "상호작용": "intrcQesitm", "이상반응": "seQesitm",
                 "보관법": "depositMethodQesitm"}
APPROVAL_F = {"seq": "ITEM_SEQ", "material": "MATERIAL_NAME", "main_ingr": "MAIN_ITEM_INGR"}
DUR_F = {"a_name": "ITEM_NAME", "b_name": "MIXTURE_ITEM_NAME", "b_seq": "MIXTURE_ITEM_SEQ",
         "a_ingr": "INGR_KOR_NAME", "b_ingr": "MIXTURE_INGR_KOR_NAME", "reason": "PROHBT_CONTENT"}

# 시스템 프롬프트는 베이스라인·SFT·DPO·서비스 모두 이 하나만 쓴다 (학습 때와 추론 때가 다르면 비교가 무의미해진다)
SYSTEM_PROMPT = (
    "당신은 처방약 복약 안내 도우미입니다. [자료]에 있는 내용만 근거로 짧고 쉽게 답하세요. "
    "답변의 마지막 줄에는 '근거: [자료ID] 항목 — \"자료 원문\"' 형식으로 자료의 문장을 한 글자도 바꾸지 말고 인용하세요. "
    "자료에 없거나 확실하지 않으면 추측하지 말고 '잘 모르겠습니다'라고 솔직히 답한 뒤 약사 또는 의사와 상담하도록 안내하세요 "
    "(이때 마지막 줄은 '근거: 제공된 자료 [자료ID]에서 해당 내용을 찾지 못했습니다.'). "
    "DUR 병용금기는 [자료]에 조회 결과가 있을 때만 단정하고, 조회 결과가 없다는 것을 '안전하다'는 뜻으로 말하지 마세요. "
    "영양제, 복용량 임의 조절, 복용 중단 판단은 답하지 않습니다."
)
UNKNOWN_TEXT = "제공된 자료에서 해당 내용을 확인할 수 없어 잘 모르겠습니다. 정확한 안내를 위해 약사 또는 의사와 상담해 주세요."
UNKNOWN_PATTERN = re.compile(r"(잘 모르겠|확인할 수 없|확인되지 않|찾지 못)")
MAX_ANSWER_CHARS, MAX_QUOTE_CHARS = 160, 200


# ---------- 공공데이터 API ----------
class ApiError(RuntimeError):
    pass


def _api_key(env_name: str) -> str:
    key = os.getenv(env_name, "").strip()
    if not key:
        raise RuntimeError(f"{env_name} 환경변수를 먼저 설정하세요.")
    return key


def mfds_get(base: str, operation: str, key: str, params: dict[str, Any], retries: int = 3) -> dict[str, Any]:
    """data.go.kr 호출. 실패 메시지에 요청 URL(=서비스키)이 섞이지 않도록 예외를 직접 정리한다. 429/5xx는 재시도."""
    query = {"serviceKey": key, "type": "json", **params}
    last = ""
    for attempt in range(retries):
        try:
            r = requests.get(f"{base}/{operation}", params=query, timeout=45)
        except requests.RequestException as exc:
            last = type(exc).__name__          # str(exc)에는 URL이 들어 있어 쓰지 않는다
            time.sleep(1.5 * (attempt + 1))
            continue
        if r.status_code in (429, 500, 502, 503, 504):
            last = f"HTTP {r.status_code}"
            time.sleep(1.5 * (attempt + 1))
            continue
        if r.status_code != 200:
            raise ApiError(f"HTTP {r.status_code}: {re.sub(r'<[^>]+>', ' ', r.text)[:200]}")
        try:
            payload = r.json()
        except ValueError:
            # 키 미승인·트래픽 초과 등 포털 오류는 type=json이어도 XML로 온다
            raise ApiError("JSON이 아닌 응답: " + re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", r.text))[:200])
        root = payload.get("response", payload)
        header = root.get("header", {}) or {}
        code = str(header.get("resultCode", "00"))
        if code not in ("00", "0", "None"):
            raise ApiError(f"MFDS API 오류 {code}: {header.get('resultMsg', '')}")
        return root
    raise ApiError(f"재시도 후에도 실패: {last}")


def items_of(root: dict[str, Any]) -> tuple[list[dict[str, Any]], int]:
    body = root.get("body", {}) or {}
    items = body.get("items", []) or []
    if isinstance(items, dict):
        items = items.get("item", [])
    if isinstance(items, dict):
        items = [items]
    return [i for i in (items or []) if isinstance(i, dict)], int(body.get("totalCount", 0) or 0)


def fetch_all(base, op, key, params=None, page_size=100, max_pages=None, label=""):
    rows, page, total = [], 1, None
    while max_pages is None or page <= max_pages:
        got, count = items_of(mfds_get(base, op, key, {"pageNo": page, "numOfRows": page_size, **(params or {})}))
        total = count if total is None else total
        if not got:
            break
        rows += got
        print(f"{label} {len(rows):,}" + (f" / {total:,}" if total else "") + "건")
        if total and len(rows) >= total:
            break
        page += 1
        time.sleep(0.12)
    return rows


def probe_operation(base: str, candidates: list[str], key: str, params: dict[str, Any] | None = None):
    """어떤 오퍼레이션 이름이 실제로 되는지 한 번에 확인 (이름이 틀리면 404/오류가 난다)."""
    for op in candidates:
        try:
            rows, total = items_of(mfds_get(base, op, key, {"pageNo": 1, "numOfRows": 1, **(params or {})}, retries=1))
            print(f"OK   {op}: totalCount={total}, 필드={list(rows[0])[:12] if rows else '-'}")
        except Exception as exc:  # noqa: BLE001
            print(f"FAIL {op}: {str(exc)[:120]}")


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


# ---------- 텍스트/약 문서 ----------
def norm(s): return re.sub(r"\s+", " ", s or "").strip()
def key_of(s): return re.sub(r"[\s\W_]+", "", s or "").lower()
def numbers(s): return set(re.findall(r"\d+(?:\.\d+)?", s or ""))


def clean_text(text: Any) -> str:
    v = html.unescape(str(text or ""))
    v = re.sub(r"<\s*br\s*/?\s*>", "\n", v, flags=re.I)
    v = re.sub(r"<[^>]+>", " ", v)
    return re.sub(r"[ \t]+", " ", v).strip()


def parse_material(text):
    """허가정보 MATERIAL_NAME: '총량 : 1정 중|성분명 : 아세트아미노펜|분량 : 500|단위 : 밀리그램;…' -> 성분·함량 목록"""
    out = []
    for entry in re.split(r";|\n", text or ""):
        kv = {}
        for part in entry.split("|"):
            if ":" in part:
                k, v = part.split(":", 1)
                kv[k.strip()] = v.strip()
        if kv.get("성분명"):
            out.append((kv["성분명"], f"{kv.get('분량', '')}{kv.get('단위', '')}".strip()))
    return out


def ingredients_from_name(name):
    """허가정보가 없을 때 제품명의 (괄호 성분)과 '50밀리그램'을 읽는다 (약한 근거)."""
    inner = [i.strip() for i in re.findall(r"\(((?:[^()]|\([^()]*\))*)\)", name or "") if i.strip() and not i.startswith("수출명")]
    base = re.sub(r"\(((?:[^()]|\([^()]*\))*)\)", "", name or "")
    amt = re.findall(r"(\d+(?:\.\d+)?)\s*(밀리그램|마이크로그램|그램|밀리리터|mg|㎎|μg|g|ml|IU|%)", base)
    if not inner and not amt:
        return []
    return [(inner[-1] if inner else "", "".join(amt[0]) if len(amt) == 1 else "")]


def make_drug_documents(easy_rows, approval_rows=None):
    """약 1개 = 문서 1개. 항목(sections) 단위로 인용한다. 허가정보는 품목기준코드(ITEM_SEQ)로 연결한다 (이름 매칭 X)."""
    appr = {str(r.get(APPROVAL_F["seq"], "")).strip(): r for r in (approval_rows or [])}
    docs = []
    for row in easy_rows:
        name, seq = str(row.get("itemName", "")).strip(), str(row.get("itemSeq", "")).strip()
        if not name or not seq:
            continue
        ap = appr.get(seq, {})
        ingr = parse_material(ap.get(APPROVAL_F["material"])) if ap else []
        src = "permit"
        if not ingr and ap.get(APPROVAL_F["main_ingr"]):
            ingr = [(re.sub(r"\[[^\]]*\]", "", p).strip(), "") for p in re.split(r"[/;,]", ap[APPROVAL_F["main_ingr"]]) if p.strip()]
        if not ingr:
            ingr, src = ingredients_from_name(name), "name"
        sections = {"제품 정보": f"제품명: {name}\n제조사: {row.get('entpName', '')}".strip()}
        if ingr:
            sections["성분·함량"] = "; ".join(" ".join(x for x in p if x) for p in ingr) + ("" if src == "permit" else " (제품명 기준)")
        for label, field in EASY_SECTIONS.items():
            t = clean_text(row.get(field, ""))
            if t:
                sections[label] = t
        if len(sections) <= 2:
            continue
        text = f"[{name}] ({row.get('entpName', '')})\n" + "\n".join(f"## {k}\n{v}" for k, v in sections.items())
        docs.append({"chunk_id": seq, "item_seq": seq, "item_name": name, "entp_name": row.get("entpName", ""),
                     "ingredients": ingr, "sections": sections, "text": text, "source_url": SOURCE_URL.format(item_seq=seq)})
    return docs


class DrugStore:
    def __init__(self, docs):
        self.by_seq = {d["item_seq"]: d for d in docs}
        self.name_count = {}
        for d in docs:
            self.name_count[key_of(d["item_name"])] = self.name_count.get(key_of(d["item_name"]), 0) + 1
        self._keys = [(key_of(d["item_name"]), d["item_seq"]) for d in docs]

    def get(self, seq): return self.by_seq.get(str(seq))

    def candidates(self, text, n=5):
        """입력/OCR 글자와 닮은 약 후보 [(item_seq, name, score)] — 사용자에게 확인받기 위한 용도"""
        t = key_of(text)
        if not t:
            return []
        scored = []
        for k, seq in self._keys:
            if k:
                m = SequenceMatcher(None, k, t, autojunk=False)
                scored.append((m.find_longest_match(0, len(k), 0, len(t)).size / len(k), m.ratio(), seq))
        scored.sort(reverse=True)
        return [(seq, self.by_seq[seq]["item_name"], round(c, 3)) for c, _r, seq in scored[:n]]


def candidates_from_ocr(store, text, n=5):
    best = {}
    for chunk in [text] + [l for l in text.split("\n") if l.strip()]:
        for seq, name, sc in store.candidates(chunk, n):
            if sc > best.get(seq, ("", -1))[1]:
                best[seq] = (name, sc)
    return [(s, nm, sc) for s, (nm, sc) in sorted(best.items(), key=lambda kv: -kv[1][1])[:n]]


def ocr_image(image_path, lang="kor+eng"):
    """로컬 Tesseract (사진이 외부로 나가지 않음). 포장 글자는 인식률이 낮을 수 있어 항상 사용자 확인을 거친다."""
    from PIL import Image, ImageOps
    import pytesseract
    img = ImageOps.autocontrast(ImageOps.grayscale(Image.open(image_path)))
    return pytesseract.image_to_string(img, lang=lang).strip()


# ---------- DUR 병용금기 ----------
def dur_docs_for(dur_rows, a_doc, b_doc):
    """a 약의 DUR 행들 중 b 약과 맞는 것을 찾는다. 1) 품목코드 2) 성분명. 없으면 '조회했지만 없음' 자료를 돌려준다."""
    hits, level = [], None
    b_names = {key_of(n) for n, _ in b_doc["ingredients"] if n}
    for r in dur_rows:
        if str(r.get(DUR_F["b_seq"], "")).strip() == b_doc["item_seq"]:
            hits.append(r); level = "품목코드"
    if not hits and b_names:
        for r in dur_rows:
            rn = key_of(r.get(DUR_F["b_ingr"], ""))
            if rn and any(rn == n or (len(n) >= 3 and (n in rn or rn in n)) for n in b_names):
                hits.append(r); level = "성분명"
    if hits:
        reason = norm(hits[0].get(DUR_F["reason"], "")) or "병용금기"
        return {"chunk_id": f"dur-{a_doc['item_seq']}-{b_doc['item_seq']}", "sections": {"DUR 병용금기": reason},
                "text": f"[DUR 병용금기 · 매칭: {level}] {a_doc['item_name']} + {b_doc['item_name']}\n## DUR 병용금기\n{reason}"}
    msg = "조회 결과 이 조합은 DUR 병용금기 자료에 없습니다."
    return {"chunk_id": f"dur-none-{b_doc['item_seq']}", "sections": {"DUR 병용금기": msg},
            "text": f"[DUR 병용금기 조회] {a_doc['item_name']} + {b_doc['item_name']}\n## DUR 병용금기\n{msg}"}


def fetch_dur_rows(dur_key, item_seq):
    # !! 조회 파라미터명(itemSeq)은 DUR 참고문서와 대조 필요. 여러 페이지를 모두 받는다 (한 페이지만 보면 조합을 놓친다).
    return fetch_all(DUR_BASE, DUR_OP, dur_key, {"itemSeq": item_seq}, label="DUR")


# ---------- 프롬프트 / 답변 형식 / 인용 검증 ----------
def render_user(drug_names, other_names, docs, question):
    ctx = "\n\n".join(f"[{d['chunk_id']}] {d['text']}" for d in docs) if docs else "(검색된 자료 없음)"
    return (f"[확인된 약]\n{', '.join(drug_names)}\n\n[복용 중인 다른 약]\n{', '.join(other_names) if other_names else '(없음)'}\n\n"
            f"[자료]\n{ctx}\n\n[질문]\n{question}")


def build_messages(drug_names, other_names, docs, question, answer=None):
    m = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": render_user(drug_names, other_names, docs, question)}]
    return m + ([{"role": "assistant", "content": answer}] if answer is not None else [])


def trim_quote(text, limit=MAX_QUOTE_CHARS):
    t = norm(text)
    if len(t) <= limit:
        return t
    cut = max(t.rfind(".", 0, limit), t.rfind(",", 0, limit), t.rfind(" ", 0, limit))
    return t[: cut + 1 if cut > 20 else limit].strip()


def first_sentence(text): return re.split(r"(?<=[.。])\s+", norm(text))[0]
def evidence_line(doc_id, section, quote): return f'근거: [{doc_id}] {section} — "{norm(quote)}"'
def unknown_answer(doc_id): return f"{UNKNOWN_TEXT}\n근거: 제공된 자료 [{doc_id}]에서 해당 내용을 찾지 못했습니다."
def says_unknown(a): return bool(UNKNOWN_PATTERN.search(a or ""))

QUOTE_LINE = re.compile(r'^근거:\s*\[([^\]]+)\]\s*(.+?)\s*—\s*"(.+)"\s*$', re.S)
NONE_LINE = re.compile(r"^근거:\s*제공된 자료 \[([^\]]+)\]에서 해당 내용을 찾지 못했습니다\.\s*$")


def check_citation(answer, docs_by_id):
    """마지막 '근거:' 줄의 자료ID/항목/원문이 실제 자료와 일치하는지 규칙으로 판정 (LLM 불필요)."""
    lines = [l.strip() for l in (answer or "").split("\n") if l.strip().startswith("근거:")]
    r = lambda kind, ok, why, doc=None: {"kind": kind, "valid": ok, "reason": why, "doc": doc}  # noqa: E731
    if not lines:
        return r("missing", False, "no_evidence_line")
    m = NONE_LINE.match(lines[-1])
    if m:
        return r("none", m.group(1) in docs_by_id, "ok" if m.group(1) in docs_by_id else "unknown_doc", m.group(1))
    m = QUOTE_LINE.match(lines[-1])
    if not m:
        return r("missing", False, "bad_format")
    doc_id, section, quote = m.group(1), m.group(2).strip(), m.group(3)
    doc = docs_by_id.get(doc_id)
    if doc is None:
        return r("quote", False, "unknown_doc", doc_id)
    text = doc["sections"].get(section)
    if text is None:
        return r("quote", False, "unknown_section", doc_id)
    return r("quote", norm(quote) in norm(text), "ok" if norm(quote) in norm(text) else "quote_not_in_section", doc_id)


# ---------- Qwen3 로드/생성 ----------
def load_qwen(model_id=MODEL_ID, adapter=None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tok = AutoTokenizer.from_pretrained(model_id)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map={"": 0}, torch_dtype=dtype)
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
    return model.eval(), tok


def make_generator(model, tok, max_new_tokens=500):
    import torch

    def generate(messages):
        # Qwen3는 기본이 '생각' 모드(<think>…)라서 끄지 않으면 토큰을 생각에 다 쓰고 답이 잘린다
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.inference_mode():   # 반복 억제(repetition_penalty)는 원문 그대로 인용하는 것을 방해하므로 쓰지 않는다
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return generate


def answer_question(generate, store, item_seq, question, others=(), dur_rows_by_other=None, dur_loaded=False):
    """확인된 약 + 질문 -> 답변. 근거 줄이 자료와 맞지 않으면 모델 답을 버리고 '잘 모르겠습니다'로 대체한다."""
    doc = store.get(item_seq)
    if doc is None:
        return {"status": "no_doc", "answer": "해당 약의 공식 자료를 찾지 못했습니다. 약사에게 확인해 주세요."}
    docs = [doc]
    other_docs = [store.get(o) for o in others if store.get(o)]
    if dur_loaded:
        for od in other_docs:
            docs.append(dur_docs_for((dur_rows_by_other or {}).get(od["item_seq"], []), doc, od))
    msgs = build_messages([doc["item_name"]], [o["item_name"] for o in other_docs], docs, question)
    raw = generate(msgs)
    by_id = {d["chunk_id"]: d for d in docs}
    chk = check_citation(raw, by_id)
    final = raw if chk["valid"] else unknown_answer(doc["chunk_id"])
    notes = []
    if other_docs and not dur_loaded:
        notes.append("※ DUR 병용금기 자료를 조회하지 않아 병용 여부는 확인하지 못했습니다. 약사에게 확인해 주세요.")
    cited = by_id.get(chk["doc"]) if chk["valid"] else None
    if cited is doc:
        notes.append(f"출처: 식약처 e약은요/허가정보 — {doc['source_url']}")
    elif cited:
        notes.append("출처: 식약처 DUR 병용금기")
    return {"status": "ok", "answer": final + ("\n" + "\n".join(notes) if notes else ""), "raw": raw, "citation": chk,
            "model_answer_replaced": not chk["valid"], "docs": [d["chunk_id"] for d in docs], "messages": msgs}

### QA·평가 코드

약 분할(`split_drugs`), QA 생성(`make_qa`), 평가 실행·지표(`run_eval`, `summarize`, `find_failures`, `evaluate`)입니다. **공통 코드 셀 다음에 이 셀을 실행해야** 아래 3번 이후 셀이 동작합니다.

In [ ]:
# ---------- QA 데이터: 약 문서에서 규칙으로 만든다 (외부 모델 출력 사용 안 함) ----------
QTEMPLATES = {
    "usage": (["사용법"], ["이 약은 어떻게 복용하나요?", "하루에 몇 번, 언제 먹어야 하나요?", "복용 방법을 알려주세요."]),
    "caution": (["사용상 주의사항", "사용 전 주의사항"], ["이 약을 복용할 때 주의할 점은 무엇인가요?", "복용 전에 확인해야 할 점이 있나요?"]),
    "side_effect": (["이상반응"], ["이 약의 부작용은 무엇인가요?", "복용 중 이상 증상이 나타나면 어떻게 하나요?"]),
    "interaction": (["상호작용"], ["다른 약과 함께 복용할 때 주의할 점이 있나요?", "같이 먹으면 안 되는 약이나 음식이 있나요?"]),
    "storage": (["보관법"], ["이 약은 어떻게 보관해야 하나요?"]),
    "unanswerable": ([], ["이 약의 가격과 보험 적용 여부를 알려주세요.", "제 지병이 있는데 이 약을 먹어도 되는지 알려주세요.",
                          "이 약과 다른 제조사 약 중 어느 것이 더 효과가 좋은가요?"]),
}
EVAL_TYPES = ["usage", "caution", "side_effect", "interaction", "unanswerable"]   # 약 1개당 5문항


def gold_answer(doc, qtype):
    """정답은 자료 문장을 그대로 인용하는 추출형이다. (형식·근거 인용·거절 습관을 가르치는 용도. 의역 능력은 가르치지 못한다.)
    자료에 해당 항목이 없으면 None."""
    if qtype == "unanswerable":
        return unknown_answer(doc["chunk_id"])
    for label in QTEMPLATES[qtype][0]:
        if label in doc["sections"]:
            sent = trim_quote(first_sentence(doc["sections"][label]), MAX_ANSWER_CHARS)
            if len(sent) >= 8 and numbers(sent) <= numbers(doc["text"]):
                return f"{sent}\n" + evidence_line(doc["chunk_id"], label, sent)
    return None


def make_qa(docs, types, seed=0):
    rng = random.Random(seed)
    out = []
    for d in docs:
        for t in types:
            g = gold_answer(d, t)
            if g is None:
                continue
            out.append({"id": f"{d['item_seq']}-{t}", "item_seq": d["item_seq"], "type": t,
                        "question": rng.choice(QTEMPLATES[t][1]), "gold": g})
    return out


def split_drugs(docs, n_eval=10, n_pool=30, n_train=None, seed=7):
    """약 단위로 분리: 평가 / DPO 후보 수집(pool) / SFT 학습. 같은 약(같은 제품명)이 둘로 갈라지지 않게 이름으로 묶는다."""
    rng = random.Random(seed)
    groups = {}
    for d in docs:
        groups.setdefault(key_of(d["item_name"]), []).append(d)
    keys = list(groups)
    rng.shuffle(keys)
    pick = lambda ks: [d for k in ks for d in groups[k]]  # noqa: E731
    ev, pool = keys[:n_eval], keys[n_eval:n_eval + n_pool]
    train = keys[n_eval + n_pool:][:n_train] if n_train else keys[n_eval + n_pool:]
    return pick(ev), pick(pool), pick(train)


# ---------- 평가 ----------
def run_eval(generate, store, entries):
    results = []
    for i, e in enumerate(entries, 1):
        r = answer_question(generate, store, e["item_seq"], e["question"])
        c = r.get("citation") or {"kind": "missing", "valid": False, "reason": "no_doc", "doc": None}
        results.append({"id": e["id"], "type": e["type"], "item_seq": e["item_seq"], "question": e["question"],
                        "answer": r.get("raw", r["answer"]), "citation": c, "says_unknown": says_unknown(r.get("raw", "")),
                        "answer_chars": len(r.get("raw", "").split("\n근거:")[0]),
                        "prompt_messages": r.get("messages")})
        if i % 10 == 0:
            print(f"  {i}/{len(entries)}")
    return results


def _mean(xs):
    xs = list(xs)
    return round(sum(xs) / len(xs), 3) if xs else None


def summarize(results):
    ans = [r for r in results if r["type"] != "unanswerable"]
    una = [r for r in results if r["type"] == "unanswerable"]
    return {"n": len(results),
            "evidence_line_rate": _mean(r["citation"]["kind"] != "missing" for r in results),
            "citation_valid_rate": _mean(r["citation"]["valid"] for r in results),
            "unknown_rate_unanswerable": _mean(r["says_unknown"] for r in una),   # 높을수록 좋음
            "over_refusal_rate": _mean(r["says_unknown"] for r in ans),           # 낮을수록 좋음
            "avg_answer_chars": _mean(r["answer_chars"] for r in results)}


def find_failures(results):
    out = []
    for r in results:
        tags = []
        if not r["citation"]["valid"]:
            tags.append(f"citation:{r['citation']['reason']}")
        if r["type"] == "unanswerable" and not r["says_unknown"]:
            tags.append("answered_unanswerable")
        if r["type"] != "unanswerable" and r["says_unknown"]:
            tags.append("over_refusal")
        if r["answer_chars"] > MAX_ANSWER_CHARS * 2:
            tags.append("too_long")
        if tags:
            out.append({**r, "tags": tags})
    return out


def evaluate(name, store, entries, adapter=None, out_root="outputs/eval", max_new_tokens=500):
    import gc, torch
    model, tok = load_qwen(MODEL_ID, adapter)
    results = run_eval(make_generator(model, tok, max_new_tokens), store, entries)
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    metrics = {"run": name, "adapter": adapter, **summarize(results)}
    out = Path(out_root) / name
    out.mkdir(parents=True, exist_ok=True)
    save_jsonl(results, out / "results.jsonl")
    save_jsonl(find_failures(results), out / "failures.jsonl")
    (out / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    return results

## 1. API 키 입력

공공데이터포털 일반 인증키(Decoding)를 입력하세요. 키는 화면에 출력되지 않습니다.

In [ ]:
import getpass, os
os.environ["MFDS_EASY_KEY"] = getpass.getpass("e약은요 API 키: ")
os.environ["MFDS_APPROVAL_KEY"] = getpass.getpass("허가정보 API 키 (없으면 Enter): ")
os.environ["MFDS_DUR_KEY"] = getpass.getpass("DUR API 키 (없으면 Enter): ")

### (선택) 서비스/오퍼레이션 이름 확인

포털 버전 번호가 바뀌면 호출이 실패합니다. 아래 셀이 후보 이름을 하나씩 호출해 어느 것이 되는지, 응답 필드가 무엇인지 보여줍니다. 결과를 보고 공통 코드 위쪽의 `*_BASE/*_OP`와 필드 매핑을 고치세요.

In [ ]:
probe_operation(EASY_BASE, [EASY_OP], _api_key("MFDS_EASY_KEY"))
if os.getenv("MFDS_APPROVAL_KEY"):
    for svc, ops in {"DrugPrdtPrmsnInfoService08": ["getDrugPrdtPrmsnInq08", "getDrugPrdtPrmsnDtlInq08"],
                     "DrugPrdtPrmsnInfoService07": ["getDrugPrdtPrmsnInq07", "getDrugPrdtPrmsnDtlInq06"]}.items():
        print(svc); probe_operation(f"https://apis.data.go.kr/1471000/{svc}", ops, _api_key("MFDS_APPROVAL_KEY"))
if os.getenv("MFDS_DUR_KEY"):
    probe_operation(DUR_BASE, [DUR_OP], _api_key("MFDS_DUR_KEY"))

## 2. e약은요 + 허가정보 수집 및 약 문서 만들기

원본은 Drive에 저장되고, 다시 실행하면 **저장된 파일을 읽습니다**(런타임이 끊겨도 전체 재수집하지 않음). 약 1개가 문서 1개이며 제품명·성분·함량·효능·사용법·주의사항·상호작용·이상반응·보관법 항목으로 나뉩니다. 허가정보는 품목기준코드로 연결합니다.

In [ ]:
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
MAX_PAGES = 3          # 시험 실행. 전체 수집은 None
easy_file, approval_file = DATA_DIR / "easy_raw.jsonl", DATA_DIR / "approval_raw.jsonl"

if easy_file.exists():
    easy_rows = load_jsonl(easy_file)
else:
    easy_rows = fetch_all(EASY_BASE, EASY_OP, _api_key("MFDS_EASY_KEY"), max_pages=MAX_PAGES, label="e약은요")
    save_jsonl(easy_rows, easy_file)

approval_rows = []
if approval_file.exists():
    approval_rows = load_jsonl(approval_file)
elif os.getenv("MFDS_APPROVAL_KEY"):
    approval_rows = fetch_all(APPROVAL_BASE, APPROVAL_OP, _api_key("MFDS_APPROVAL_KEY"), max_pages=MAX_PAGES, label="허가정보")
    save_jsonl(approval_rows, approval_file)
else:
    print("허가정보 키가 없어 성분·함량은 제품명에서만 읽습니다.")

drug_docs = make_drug_documents(easy_rows, approval_rows)
save_jsonl(drug_docs, DATA_DIR / "drug_docs.jsonl")
store = DrugStore(drug_docs)
n_permit = sum(1 for d in drug_docs if d["sections"].get("성분·함량") and "제품명 기준" not in d["sections"]["성분·함량"])
print(f"e약은요 {len(easy_rows):,}건 / 허가정보 {len(approval_rows):,}건 -> 약 문서 {len(drug_docs):,}개 (성분·함량을 허가정보로 채운 문서 {n_permit:,}개)")

In [ ]:
import pandas as pd
display(pd.DataFrame([{"품목코드": d["item_seq"], "제품명": d["item_name"], "성분·함량": d["sections"].get("성분·함량", ""),
                       "항목": ", ".join(k for k in d["sections"] if k not in ("제품 정보", "성분·함량"))} for d in drug_docs[:10]]))
print(drug_docs[0]["text"][:900] if drug_docs else "문서 없음")

## 3. QA 데이터와 평가 세트 만들기

약 단위로 **평가 / DPO 후보 수집(pool) / SFT 학습**을 겹치지 않게 나눕니다. 질문은 고정 템플릿, 정답은 약 문서의 문장을 그대로 인용한 추출형(코드가 만들어서 외부 모델 출력이 학습에 들어가지 않음)입니다. 자료에 없는 질문(가격·지병 등)의 정답은 "잘 모르겠습니다 + 약사/의사 상담"입니다.

> 이 정답은 *형식·근거 인용·거절 습관*을 가르치는 용도이고, 자연스러운 의역 능력은 가르치지 못합니다. 의역형 정답이 필요하면 사람이 직접 쓰거나 사용 조건을 확인한 모델로 만드세요.

In [ ]:
eval_docs, pool_docs, train_docs = split_drugs(drug_docs, n_eval=10, n_pool=30, seed=7)
print(f"약 분할: 평가 {len(eval_docs)} / DPO pool {len(pool_docs)} / SFT {len(train_docs)}")
if len(train_docs) < 50:
    print("[주의] SFT용 약이 적습니다 (시험 실행 규모). 수집량을 늘리세요.")

eval_entries = make_qa(eval_docs, EVAL_TYPES, seed=1)
save_jsonl(eval_entries, DATA_DIR / "eval_set.jsonl")

train_entries = make_qa(train_docs, EVAL_TYPES + ["storage"], seed=2)
by_seq = {d["item_seq"]: d for d in train_docs}
sft_rows = [{"id": e["id"], "type": e["type"],
             "prompt_messages": build_messages([by_seq[e["item_seq"]]["item_name"]], [], [by_seq[e["item_seq"]]], e["question"]),
             "answer": e["gold"]} for e in train_entries]
save_jsonl(sft_rows, DATA_DIR / "sft_train.jsonl")
print(f"평가 질문 {len(eval_entries)}개, SFT 예제 {len(sft_rows)}개")
print("\n--- SFT 예제 1개 ---\n", sft_rows[0]["answer"] if sft_rows else "")

## 4~7. 텍스트 또는 사진 → 약 확인 → e약은요/DUR 검색 → Qwen 답변

OCR 글자만으로 약을 확정하지 않습니다. 후보(제품명·성분·함량)를 보여주고 **사용자가 확인한 뒤** 검색합니다. 병용 질문이면 함께 복용하는 약도 같은 방식으로 확인한 뒤 DUR 병용금기를 조회합니다. 모델이 근거 줄을 틀리게 쓰면 그 답은 버리고 "잘 모르겠습니다"로 바꿉니다.

In [ ]:
def show_candidate(i, seq, name, score):
    d = store.get(seq)
    print(f"{i}. {name} ({d['entp_name']}) | 성분·함량: {d['sections'].get('성분·함량', '확인 불가')} | 품목코드 {seq} | 유사도 {score}")


def pick_drug(prompt):
    """텍스트/사진으로 약을 찾고 사용자가 확인한 품목코드를 돌려준다."""
    mode = input(f"{prompt} 입력 방식 (1=텍스트, 2=사진): ").strip()
    if mode == "2":
        try:
            from google.colab import files
            image_path = next(iter(files.upload()))
        except ImportError:
            image_path = input("이미지 경로: ").strip()
        text = ocr_image(image_path)
        print("OCR 결과(틀릴 수 있음):\n", text)
    else:
        text = input("제품명: ").strip()
    cands = candidates_from_ocr(store, text, n=5) if mode == "2" else store.candidates(text, n=5)
    if not cands:
        return None
    for i, c in enumerate(cands, 1):
        show_candidate(i, *c)
    if len(cands) > 1 and cands[0][2] - cands[1][2] < 0.05:
        print("※ 비슷한 후보가 여러 개입니다. 제조사·성분·함량을 꼭 확인하세요.")
    n = int(input("맞는 번호 (0=맞는 약 없음): ") or 0)
    return cands[n - 1][0] if 1 <= n <= len(cands) else None


item_seq = pick_drug("[복용 약]")
assert item_seq, "약을 확인하지 못해 중단합니다."
question = input("질문: ").strip()
others, dur_rows_by_other = [], {}
if input("함께 복용 중인 다른 약이 있나요? (y/N): ").strip().lower() == "y":
    o = pick_drug("[다른 약]")
    if o:
        others.append(o)
dur_key = os.getenv("MFDS_DUR_KEY", "").strip()
if others and dur_key:
    rows = fetch_dur_rows(dur_key, item_seq)
    dur_rows_by_other = {o: rows for o in others}
qwen_model, qwen_tok = load_qwen(MODEL_ID)
res = answer_question(make_generator(qwen_model, qwen_tok), store, item_seq, question, others, dur_rows_by_other, dur_loaded=bool(dur_key))
print("\n" + res["answer"])
del qwen_model, qwen_tok
import gc, torch; gc.collect(); torch.cuda.empty_cache()

## 8. 고정 평가 세트와 베이스라인 (Qwen3-8B + RAG, QLoRA/DPO 없음)

이후 SFT/DPO와 **같은 질문·같은 시스템 프롬프트·같은 지표**로 평가합니다. 지표는 코드가 자동 계산합니다: 근거 줄 비율, 근거 일치율(자료ID·항목·원문), 자료에 없는 질문에서 모른다고 한 비율, 답할 수 있는데 거절한 비율, 답변 길이. 사진 질문은 직접 찍은 사진이 필요해 자동 평가에서 빼고, 아래에서 OCR 후보 정확도만 따로 확인합니다.

In [ ]:
baseline_results = evaluate("baseline", store, eval_entries)

## 9. 검토한 QA로 SFT용 QLoRA

2 epoch, 최대 길이 2048. 정답 부분(completion)에만 loss를 겁니다. 끝나면 4-bit/QLoRA 메모리 절감 리포트를 출력합니다. `data/sft_train.jsonl`을 열어 정답 몇 개를 직접 확인한 뒤 실행하세요.

In [ ]:
"""QLoRA SFT (Qwen3-8B). 입력: data/sft_train.jsonl (prompt_messages, answer)  — 추론과 같은 시스템 프롬프트·같은 자료 형식."""
import gc
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer


class LossPlateauStop(TrainerCallback):
    """학습 loss의 이동평균이 min_delta 이상 좋아지지 않는 로그가 patience번 연속이면 학습을 중단한다."""
    def __init__(self, window=10, patience=10, min_delta=0.005, warmup_steps=20):
        self.window, self.patience, self.min_delta, self.warmup = window, patience, min_delta, warmup_steps
        self.hist, self.best, self.bad = [], float("inf"), 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:          # 평가/종료 로그는 건너뜀
            return
        self.hist.append(logs["loss"])
        if len(self.hist) < self.window or state.global_step < self.warmup:
            return
        avg = sum(self.hist[-self.window:]) / self.window
        if avg < self.best - self.min_delta:
            self.best, self.bad = avg, 0             # 의미 있게 좋아짐 -> 카운트 초기화
        else:
            self.bad += 1
            if self.bad >= self.patience:
                print(f"[조기 종료] step {state.global_step}: loss 이동평균 {avg:.4f}가 {self.patience}회 연속 개선 없음 (최고 {self.best:.4f})")
                control.should_training_stop = True


SFT_OUT = "artifacts/qwen3-8b-med-sft"
rows = load_jsonl("data/sft_train.jsonl")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# 프롬프트는 추론 때와 똑같이(enable_thinking=False) 직접 렌더링하고, 정답 부분(completion)에만 loss를 건다.
# (assistant_only_loss=True는 채팅 템플릿에 {% generation %} 표시가 있어야 해서 Qwen3 기본 템플릿에서는 동작하지 않는다)
ds = Dataset.from_list([{
    "prompt": tok.apply_chat_template(r["prompt_messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False),
    "completion": r["answer"] + tok.eos_token} for r in rows])
lens = [len(tok(p + c, add_special_tokens=False)["input_ids"]) for p, c in zip(ds["prompt"], ds["completion"])]
print(f"SFT 예제 {len(ds)}개, 토큰 길이 max={max(lens)} / 평균={sum(lens)//len(lens)}")
MAX_LEN = 5000   # 자료(약 문서) + 질문 + 답이 들어가야 한다 (잘리면 근거 줄이 사라진다)
assert max(lens) <= MAX_LEN, "일부 예제가 max_length를 넘습니다. 문서 길이를 줄이거나 MAX_LEN을 늘리세요."

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, torch_dtype=dtype)
model.config.use_cache = False
mem_4bit_gib = model.get_memory_footprint() / 1024**3
model = prepare_model_for_kbit_training(model)

args = SFTConfig(output_dir=SFT_OUT, num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=8,
                 learning_rate=2e-4, max_length=MAX_LEN, gradient_checkpointing=True, optim="paged_adamw_8bit",
                 bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16), logging_steps=1, save_strategy="epoch",
                 report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tok,
                     callbacks=[LossPlateauStop(window=10, patience=10, min_delta=0.005, warmup_steps=20)],
                     peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                                            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
trainer.train()
trainer.save_model(SFT_OUT); tok.save_pretrained(SFT_OUT)

# ----- 메모리 절감 리포트 (전체 파인튜닝을 실제로 돌릴 수는 없어서 '추정'과 '실측'을 구분해 표시) -----
from accelerate import init_empty_weights
from transformers import AutoConfig
with init_empty_weights():
    n_params = sum(p.numel() for p in AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(MODEL_ID)).parameters())
peak = torch.cuda.max_memory_allocated() / 1024**3
bf16_gib = n_params * 2 / 1024**3
full_ft_gib = n_params * 12 / 1024**3      # bf16 가중치 2B + 그래디언트 2B + Adam 상태 8B (활성값 제외한 하한 추정)
lora_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"파라미터 {n_params/1e9:.2f}B | 학습 파라미터(LoRA) {lora_params/1e6:.1f}M ({100*lora_params/n_params:.2f}%)")
print(f"모델 가중치: bf16 {bf16_gib:.1f} GiB -> 4-bit {mem_4bit_gib:.1f} GiB (실측, {100*(1-mem_4bit_gib/bf16_gib):.0f}% 감소)")
print(f"전체 파인튜닝 하한 추정 {full_ft_gib:.1f} GiB(추정) vs QLoRA 학습 중 최대 {peak:.1f} GiB(실측) -> 약 {100*(1-peak/full_ft_gib):.0f}% 적음")
print("저장:", SFT_OUT)
del trainer, model; gc.collect(); torch.cuda.empty_cache()

### SFT 평가

In [ ]:
sft_results = evaluate("sft", store, eval_entries, adapter=SFT_OUT)

## 10. 반복 문제를 고친 DPO (선택)

SFT 평가에서 **같은 문제가 반복될 때만** 실행합니다. 평가 세트와 겹치지 않는 pool 약으로 SFT 모델을 돌려 실패 답변(rejected)과 검증된 정답(chosen)을 짝짓습니다. 선호 기준은 근거 일치·자료에 없는 것을 아는 척하지 않기·불필요한 거절 금지·간결성입니다.

In [ ]:
"""SFT 평가에서 '반복되는' 실패만 골라 DPO 선호 쌍을 만들고 학습한다.
  pool 약(평가·SFT와 겹치지 않음)으로 SFT 모델을 돌려 -> 실패 답변 = rejected, 같은 질문의 검증된 정답 = chosen"""
import gc
from collections import Counter

MIN_REPEAT = 3                                            # 같은 문제 유형이 이 횟수 이상 반복될 때만 학습 대상으로 삼는다
pool_entries = make_qa(pool_docs, EVAL_TYPES, seed=3)
gold_by_id = {e["id"]: e["gold"] for e in pool_entries}
pool_results = evaluate("sft_pool", store, pool_entries, adapter=SFT_OUT)
fails = find_failures(pool_results)
tag_count = Counter(t for f in fails for t in set(f["tags"]))
print("반복 문제 집계:", dict(tag_count))
repeat_tags = {t for t, c in tag_count.items() if c >= MIN_REPEAT}
pairs = [{"id": f["id"], "tags": f["tags"], "prompt_messages": f["prompt_messages"], "chosen": gold_by_id[f["id"]], "rejected": f["answer"]}
         for f in fails if repeat_tags & set(f["tags"]) and f["answer"].strip() != gold_by_id[f["id"]].strip()]
save_jsonl(pairs, "data/dpo_train.jsonl")
print(f"DPO 선호 쌍 {len(pairs)}개 (반복 태그: {sorted(repeat_tags)})")

if len(pairs) < 20:
    print("[주의] 쌍이 20개 미만입니다. DPO는 보통 이보다 많은 쌍이 필요합니다. pool 약을 늘린 뒤 다시 실행하세요.")
if not pairs:
    print("반복되는 문제가 없어 DPO를 건너뜁니다.")
else:
    import torch
    from datasets import Dataset
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from trl import DPOConfig, DPOTrainer

    DPO_OUT = "artifacts/qwen3-8b-med-dpo"
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    ds = Dataset.from_list([{
        "prompt": tok.apply_chat_template(p["prompt_messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False),
        "chosen": p["chosen"] + tok.eos_token, "rejected": p["rejected"] + tok.eos_token} for p in pairs])
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, torch_dtype=dtype)
    base.config.use_cache = False
    # 기준(reference) 모델은 'SFT를 마친 모델'이어야 한다. ref_model=None + 어댑터 비활성화는 SFT 이전의 베이스 모델이 기준이 되어 버린다.
    # 그래서 같은 SFT 어댑터를 두 번 올려서 하나(train)만 학습하고 하나(ref)는 고정한다.
    # !! model_adapter_name/ref_adapter_name 옵션은 설치된 TRL 버전의 DPOConfig에서 지원되는지 확인 필요
    policy = PeftModel.from_pretrained(base, SFT_OUT, adapter_name="train", is_trainable=True)
    policy.load_adapter(SFT_OUT, adapter_name="ref")
    policy.set_adapter("train")
    cfg = DPOConfig(output_dir=DPO_OUT, num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=8,
                    learning_rate=2e-5, beta=0.1, max_length=2048, gradient_checkpointing=True, optim="paged_adamw_8bit",
                    bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16), logging_steps=1, save_strategy="epoch",
                    report_to="none", model_adapter_name="train", ref_adapter_name="ref")
    trainer = DPOTrainer(model=policy, ref_model=None, args=cfg, train_dataset=ds, processing_class=tok)
    trainer.train()
    policy.save_pretrained(DPO_OUT, selected_adapters=["train"])
    tok.save_pretrained(DPO_OUT)
    # PEFT는 기본 이름이 아닌 어댑터를 하위 폴더(train/)에 저장한다 -> adapter_config.json이 있는 곳을 찾아 사용
    DPO_ADAPTER = next(str(p.parent) for p in [Path(DPO_OUT) / "adapter_config.json", Path(DPO_OUT) / "train" / "adapter_config.json"] if p.exists())
    print("DPO 어댑터 저장:", DPO_ADAPTER)
    del trainer, policy, base; gc.collect(); torch.cuda.empty_cache()

## 11. 같은 평가 세트로 전후 비교

In [ ]:
if "DPO_ADAPTER" in globals():
    dpo_results = evaluate("dpo", store, eval_entries, adapter=DPO_ADAPTER)
runs = [r for r in ("baseline", "sft", "dpo") if (Path("outputs/eval") / r / "metrics.json").exists()]
table = pd.DataFrame({r: json.loads((Path("outputs/eval") / r / "metrics.json").read_text()) for r in runs}).drop(["run", "adapter"])
display(table)

## 점수 해석

- `citation_valid_rate`: 근거 줄의 자료ID·항목·원문이 실제 자료와 일치하는 비율 (규칙 판정, 높을수록 좋음)
- `unknown_rate_unanswerable`: 자료에 없는 질문에서 솔직히 모른다고 한 비율 (높을수록 좋음)
- `over_refusal_rate`: 답할 수 있는 질문에서 거절한 비율 (낮을수록 좋음)
- 평가 질문은 50개 안팎이라 한 문항 차이가 2%p입니다. 작은 차이는 우연일 수 있으니 `outputs/eval/*/results.jsonl`의 실제 답변을 직접 읽어 보세요.
- 이 지표는 *근거 인용 형식과 거절 행동*을 봅니다. 답의 의학적 정확성·누락은 별도로 사람이 검토해야 하고, DUR 결과가 없다는 것이 안전을 뜻하지 않습니다.